# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their @id
print("Available Record Sets:")
for rs in metadata.record_sets:
    print(f"  @id: {rs.id} | name: {rs.name} | description: {rs.description}")

record_sets = metadata.record_sets
# As an example, show the fields and columns in the first record set
if record_sets:
    example_rs = record_sets[0]
    print(f"\nFields for record set {example_rs.id}:")
    for field in example_rs.fields:
        print(f"    Field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"      Column @id: {col.id} | name: {col.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare record set IDs
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Show columns for the first record set DataFrame loaded
if dataframes:
    main_rs_id = record_set_ids[0]
    print(f"Columns for record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA for a numeric field
import numpy as np

if dataframes:
    df = dataframes[main_rs_id]

    # Search for a numeric field to analyze (e.g., peptide count, coverage, MW, etc.)
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.number, 'float64', 'int64', 'float32', 'int32']]
    if not numeric_candidates:
        # Try to infer numerics: convert columns that are float but read as object
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                pass
        numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric fields: {numeric_candidates}")
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        threshold = df[numeric_field].mean() if df[numeric_field].dtype != 'O' else 10

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df[[numeric_field]].head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field
        group_field = None
        for c in df.columns:
            if df[c].dtype == 'O' and c != numeric_field:
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (average {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data available for Exploratory Data Analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualize the distribution of the selected numeric field and its normalized version
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidates:
    if len(filtered_df) > 0:
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        sns.histplot(filtered_df[numeric_field], kde=True)
        plt.title(f'Distribution of {numeric_field}')
        plt.xlabel(numeric_field)
        plt.subplot(1, 2, 2)
        sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True)
        plt.title(f'Normalized {numeric_field}')
        plt.xlabel(f'{numeric_field}_normalized')
        plt.tight_layout()
        plt.show()
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.xticks(rotation=45)
        plt.title(f'Average {numeric_field} by {group_field}')
        plt.tight_layout()
        plt.show()
else:
    print('No numeric field visualization available.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides detailed protein information from human mast cell-derived extracellular vesicles, including peptide and molecular attributes.
- Using `mlcroissant`, we can easily access its structure via `@id` and analyze fields like protein abundance or counts.
- Basic EDA and visualizations demonstrate data complexity and provide a starting point for further biological analysis, such as finding outliers in protein expression, normalization, or group-wise comparisons.

_This framework can be extended for domain-specific statistical testing, machine learning preparation, or integration with other bioinformatics sources._